# 02d — 第二讲基础习题参考答案（Solutions）

本文件对应 `02d_exercises.ipynb` 中全部 8 道监督学习基础巩固习题的完整参考代码与结果说明。

---

## 0. 公共环境与 ESOL 特征矩阵准备

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors

for p in [Path("../../01 - Basics/data/esol.csv"), Path("../01 - Basics/data/esol.csv"), Path("data/esol.csv")]:
    if p.exists():
        df = pd.read_csv(p)
        break

df = df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "logS",
})[["name", "smiles", "logS"]].copy()
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)

DESC_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "NumHDonors": Descriptors.NumHDonors,
    "NumHAcceptors": Descriptors.NumHAcceptors,
    "NumRotatableBonds": Descriptors.NumRotatableBonds,
    "RingCount": Descriptors.RingCount,
    "NumAromaticRings": Descriptors.NumAromaticRings,
    "HeavyAtomCount": Descriptors.HeavyAtomCount,
    "FractionCSP3": Descriptors.FractionCSP3,
}
X = pd.DataFrame([{k: fn(m) for k, fn in DESC_FUNCS.items()} for m in df["mol"]])
y_reg = df["logS"].values

print(f"成功准备特征矩阵 X: {X.shape} | 目标向量 y_reg: {y_reg.shape}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：回归基础与特征缩放（对应 `02a`）

### 习题 1：训练/测试集划分与“均值猜测基线”

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.25, random_state=0
)

y_pred_mean = np.full_like(y_test, fill_value=y_train.mean())
mae_base  = mean_absolute_error(y_test, y_pred_mean)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_mean))

print(f"Baseline MAE = {mae_base:.3f} | Baseline RMSE = {rmse_base:.3f}")

### 习题 2：训练多元线性回归并计算测试集 $R^2$ 与 RMSE

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression().fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
r2_lin   = r2_score(y_test, y_pred_lin)
print(f"LinearRegression Test RMSE = {rmse_lin:.3f} | Test R2 = {r2_lin:.3f}")

### 习题 3：验证特征标准化对 k 近邻回归（kNN）的提升

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

knn_raw = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
rmse_raw = np.sqrt(mean_squared_error(y_test, knn_raw.predict(X_test)))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

knn_scaled = KNeighborsRegressor(n_neighbors=5).fit(X_train_s, y_train)
rmse_scaled = np.sqrt(mean_squared_error(y_test, knn_scaled.predict(X_test_s)))

print(f"未标准化 RMSE = {rmse_raw:.3f} | 标准化后 RMSE = {rmse_scaled:.3f}")

---

## 第二部分：防泄漏 Pipeline、交叉验证与超参数调优（对应 `02b`）

### 习题 4：用 `make_pipeline` + `cross_val_score` 执行 5 折交叉验证

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

ridge_pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
cv_r2 = cross_val_score(ridge_pipe, X, y_reg, cv=5, scoring="r2")

print(f"各折 R2: {np.round(cv_r2, 3)}")
print(f"5-Fold CV R2: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

### 习题 5：使用 `GridSearchCV` 为单棵决策树寻找最优深度

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

param_grid = {"max_depth": [2, 4, 6, 8, 12, None]}
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
grid.fit(X_train, y_train)

test_rmse = np.sqrt(mean_squared_error(y_test, grid.predict(X_test)))
print(f"最优参数: {grid.best_params_} | 测试集 RMSE = {test_rmse:.3f}")

---

## 第三部分：二元分类与特征重要性（对应 `02c`）

### 习题 6：构造二分类标签并进行分层抽样划分（`stratify`）

In [ ]:
y_cls = (df["logS"] < -3.0).astype(int)
print(f"全数据集正类 (logS < -3) 占比: {y_cls.mean():.3f}")

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, stratify=y_cls, random_state=42
)
print(f"训练集正类占比: {y_tr_c.mean():.3f} | 测试集正类占比: {y_te_c.mean():.3f}")

### 习题 7：训练逻辑回归分类器并计算核心分类指标

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

clf_lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr_c, y_tr_c)
y_pred = clf_lr.predict(X_te_c)
y_prob = clf_lr.predict_proba(X_te_c)[:, 1]

print(
    f"Accuracy = {accuracy_score(y_te_c, y_pred):.3f} | "
    f"Precision = {precision_score(y_te_c, y_pred):.3f} | "
    f"Recall = {recall_score(y_te_c, y_pred):.3f} | "
    f"F1 = {f1_score(y_te_c, y_pred):.3f} | "
    f"ROC-AUC = {roc_auc_score(y_te_c, y_prob):.3f}"
)

### 习题 8：训练随机森林分类器并绘制特征重要性条形图

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_c, y_tr_c)
imp = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(6, 3.8))
imp.plot(kind="barh", color="steelblue", ax=ax)
ax.set_xlabel("MDI Feature Importance")
ax.set_title("Random Forest Feature Importances (logS < -3)")
plt.tight_layout()
plt.show()

print("排名前 3 的特征:", imp.index[::-1][:3].tolist())